In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cat = "`data-bricks-fundamentals`.default"

# ===== Gold 1: how much did food prices rise, by province (Q1-3) =====
cpi = spark.table(f"{cat}.silver_cpi_food")
latest = cpi.agg(F.max("month")).first()[0]

base   = (cpi.filter(F.year("month") == 2017)
             .groupBy("geo", "product").agg(F.avg("cpi_index").alias("index_2017")))
recent = (cpi.filter(F.col("month") > F.add_months(F.lit(latest), -12))
             .groupBy("geo", "product").agg(F.avg("cpi_index").alias("index_recent")))

change = (base.join(recent, ["geo", "product"])
    .withColumn("pct_change", F.round((F.col("index_recent") / F.col("index_2017") - 1) * 100, 1)))

canada = (change.filter(F.col("geo") == "Canada")
    .select("product", F.col("pct_change").alias("canada_pct_change")))

gold_cpi = (change.filter(F.col("geo") != "Canada")
    .join(canada, "product")
    .withColumn("pts_vs_canada", F.round(F.col("pct_change") - F.col("canada_pct_change"), 1))
    .withColumn("rank_in_product",
        F.rank().over(Window.partitionBy("product").orderBy(F.desc("pct_change")))))

gold_cpi.write.mode("overwrite").saveAsTable(f"{cat}.gold_cpi_change_by_province")

# ===== Gold 2: who pays the most today, by product (Q6, 7, 9) =====
avg = spark.table(f"{cat}.silver_avg_prices_food")
latest_avg = avg.agg(F.max("month")).first()[0]

recent_avg = (avg.filter(F.col("month") > F.add_months(F.lit(latest_avg), -3))
    .groupBy("geo", "product").agg(F.round(F.avg("avg_price"), 2).alias("price_3m")))

canada_p = (recent_avg.filter(F.col("geo") == "Canada")
    .select("product", F.col("price_3m").alias("canada_price")))

gold_price = (recent_avg.filter(F.col("geo") != "Canada")
    .join(canada_p, "product")
    .withColumn("pct_vs_canada",
        F.round((F.col("price_3m") / F.col("canada_price") - 1) * 100, 1))
    .withColumn("rank_most_expensive",
        F.rank().over(Window.partitionBy("product").orderBy(F.desc("price_3m")))))

gold_price.write.mode("overwrite").saveAsTable(f"{cat}.gold_price_by_province_latest")

In [0]:
%sql
SELECT geo, pct_change
FROM `data-bricks-fundamentals`.default.gold_cpi_change_by_province
WHERE product = 'Food' ORDER BY pct_change DESC;
-- Expect PEI at the top (45.1%) and Newfoundland and Labrador at the bottom (37.3%). Canada is 40.9%.

In [0]:
%sql
SELECT geo, price_3m
FROM `data-bricks-fundamentals`.default.gold_price_by_province_latest
WHERE product = 'Milk, 2 litres' ORDER BY price_3m DESC;
-- Expect Ontario at the top ($5.92) and PEI at the bottom ($4.73).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cat = "`data-bricks-fundamentals`.default"
cpi = spark.table(f"{cat}.silver_cpi_food")
w = Window.partitionBy("geo", "product").orderBy("month")

# ===== Gold 3: volatility (Q5) =====
monthly = cpi.withColumn("mom_pct", (F.col("cpi_index") / F.lag("cpi_index", 1).over(w) - 1) * 100)

gold_vol = (monthly.groupBy("geo", "product")
    .agg(F.round(F.stddev("mom_pct"), 2).alias("volatility"))
    .withColumn("rank_most_stable",
        F.rank().over(Window.partitionBy("geo").orderBy("volatility"))))
gold_vol.write.mode("overwrite").saveAsTable(f"{cat}.gold_cpi_volatility")

# ===== Gold 4: spike and cool-down (Q4, Q11) =====
COOL_LEVEL = 3.0   # "cooled down" = yearly increase below 3%

yoy = (cpi.withColumn("yoy_pct", F.round((F.col("cpi_index") / F.lag("cpi_index", 12).over(w) - 1) * 100, 1))
          .filter(F.col("yoy_pct").isNotNull()))

spike_window = yoy.filter((F.col("month") >= "2021-01-01") & (F.col("month") <= "2023-12-01"))
peak = (spike_window
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("geo", "product").orderBy(F.desc("yoy_pct"), F.asc("month"))))
    .filter("rn = 1")
    .select("geo", "product",
            F.col("month").alias("peak_month"), F.col("yoy_pct").alias("peak_yoy_pct")))

latest_m = yoy.agg(F.max("month")).first()[0]
now = (yoy.filter(F.col("month") == latest_m)
    .select("geo", "product", F.col("yoy_pct").alias("latest_yoy_pct")))

cooled = (yoy.join(peak, ["geo", "product"])
    .filter((F.col("month") > F.col("peak_month")) & (F.col("yoy_pct") < COOL_LEVEL))
    .groupBy("geo", "product").agg(F.min("month").alias("cooled_month")))

gold_spike = (peak.join(now, ["geo", "product"])
    .join(cooled, ["geo", "product"], "left")
    .withColumn("months_to_cool",
        F.months_between("cooled_month", "peak_month").cast("int"))
    .withColumn("rank_biggest_spike",
        F.rank().over(Window.partitionBy("geo").orderBy(F.desc("peak_yoy_pct")))))
gold_spike.write.mode("overwrite").saveAsTable(f"{cat}.gold_cpi_spike_recovery")

# ===== Gold 5: province price gap over time (Q8) =====
avg = spark.table(f"{cat}.silver_avg_prices_food").filter(F.col("geo") != "Canada")

yearly = (avg.withColumn("year", F.year("month"))
    .groupBy("product", "geo", "year")
    .agg(F.avg("avg_price").alias("yr_price"),
         F.countDistinct("month").alias("months_in_year")))

gold_gap = (yearly.groupBy("product", "year")
    .agg(F.max("yr_price").alias("hi"), F.min("yr_price").alias("lo"),
         F.max_by("geo", "yr_price").alias("highest_province"),
         F.min_by("geo", "yr_price").alias("lowest_province"),
         F.max("months_in_year").alias("months_in_year"))
    .withColumn("gap_pct", F.round((F.col("hi") / F.col("lo") - 1) * 100, 1))
    .withColumn("highest_price", F.round("hi", 2))
    .withColumn("lowest_price", F.round("lo", 2))
    .drop("hi", "lo"))
gold_gap.write.mode("overwrite").saveAsTable(f"{cat}.gold_price_gap_trend")

In [0]:
%sql
-- Q5: most and least stable food groups in Canada
SELECT product, volatility
FROM `data-bricks-fundamentals`.default.gold_cpi_volatility
WHERE geo = 'Canada' ORDER BY volatility;
-- Expect Food first (0.57) and Fresh vegetables last (3.76).

In [0]:
%sql
-- Q8: did the province gap shrink?
SELECT product, year, highest_province, lowest_province, gap_pct
FROM `data-bricks-fundamentals`.default.gold_price_gap_trend
WHERE product = 'Milk, 2 litres' AND year IN (2017, 2025);
-- Expect the gap to go from 29.5% to 25.9%.

In [0]:
%sql
-- Q11: which province cooled down fastest after the food price spike?
SELECT geo, peak_yoy_pct, peak_month, latest_yoy_pct, months_to_cool
FROM `data-bricks-fundamentals`.default.gold_cpi_spike_recovery
WHERE product = 'Food' AND geo <> 'Canada' ORDER BY months_to_cool;
-- Expect British Columbia first (15 months) and New Brunswick last (27 months).

In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cat = "`data-bricks-fundamentals`.default"

# ===== Gold 6: seasonality (Q10), Canada, full years 2017-2025 =====
avg = spark.table(f"{cat}.silver_avg_prices_food")

idx = (avg.filter((F.col("geo") == "Canada") & (F.year("month") <= 2025))
    .withColumn("year", F.year("month"))
    .withColumn("month_num", F.month("month"))
    .withColumn("ratio", F.col("avg_price") / F.avg("avg_price").over(Window.partitionBy("product", "year")))
    .groupBy("product", "month_num").agg(F.avg("ratio").alias("seasonal_index")))

idx.select("product", "month_num", F.round("seasonal_index", 3).alias("seasonal_index")) \
   .write.mode("overwrite").saveAsTable(f"{cat}.gold_seasonality")

(idx.groupBy("product").agg(
        F.round((F.max("seasonal_index") - F.min("seasonal_index")) * 100, 1).alias("swing_pct"),
        F.max_by("month_num", "seasonal_index").alias("priciest_month"),
        F.min_by("month_num", "seasonal_index").alias("cheapest_month"))
 .write.mode("overwrite").saveAsTable(f"{cat}.gold_seasonality_summary"))

# ===== Gold 7: do categories lead each other? (Q12) =====
cpi = spark.table(f"{cat}.silver_cpi_food")
w = Window.partitionBy("geo", "product").orderBy("month")
yoy = (cpi.withColumn("yoy_pct", F.round((F.col("cpi_index") / F.lag("cpi_index", 12).over(w) - 1) * 100, 1))
          .filter(F.col("yoy_pct").isNotNull()))

g7 = ["Meat", "Fish, seafood and other marine products", "Dairy products and eggs",
      "Bakery and cereal products (excluding baby food)",
      "Fruit, fruit preparations and nuts", "Vegetables and vegetable preparations",
      "Other food products and non-alcoholic beverages"]

cy = yoy.filter((F.col("geo") == "Canada") & F.col("product").isin(g7)).select("month", "product", "yoy_pct")
leader = cy.select("month", F.col("product").alias("leader"), F.col("yoy_pct").alias("leader_yoy"))

parts = []
for k in range(0, 7):   # follower is compared k months AFTER the leader
    follower = cy.select(F.add_months("month", -k).alias("month"),
                         F.col("product").alias("follower"), F.col("yoy_pct").alias("follower_yoy"))
    parts.append(leader.join(follower, "month")
        .filter(F.col("leader") != F.col("follower"))
        .groupBy("leader", "follower").agg(F.corr("leader_yoy", "follower_yoy").alias("corr"))
        .withColumn("lag_months", F.lit(k)))

gold_leadlag = (reduce(lambda a, b: a.unionByName(b), parts)
    .withColumn("rn", F.row_number().over(Window.partitionBy("leader", "follower").orderBy(F.desc("corr"))))
    .filter("rn = 1")
    .select("leader", "follower", "lag_months", F.round("corr", 2).alias("correlation")))
gold_leadlag.write.mode("overwrite").saveAsTable(f"{cat}.gold_category_lead_lag")

In [0]:
%sql
SELECT COUNT(*) FROM `data-bricks-fundamentals`.default.gold_price_pressure_watchlist WHERE on_watchlist;
-- Expect 50 (of 320 province and category combinations).

In [0]:
%sql
SELECT product, pct_vs_canada FROM `data-bricks-fundamentals`.default.gold_promo_candidates
WHERE geo = 'British Columbia' ORDER BY rank_in_province LIMIT 3;
-- Expect Potatoes 4.54 kg (+49.2%), Broccoli (+39.2%), Avocado (+26.5%).

In [0]:
%sql
SELECT DISTINCT product, canada_vs_food_pts FROM `data-bricks-fundamentals`.default.gold_relative_price_change
ORDER BY canada_vs_food_pts DESC;
-- Expect Fresh or frozen beef at the top (+38.3 points) and Cheese at the bottom (-16.0).